# Pipeline d’analyse quantitative de l’action AAPL

Ce notebook suit le pipeline suivant : **AAPL → MA_100/MA_200 → RSI → MACD → volatilité → volume → stratégie → backtesting → rendement → Sharpe Ratio → Maximum Drawdown**.

> **Avertissement pédagogique :** le backtest est une simulation historique et ne constitue pas un conseil financier. Les résultats passés ne garantissent pas les résultats futurs.

## 1. Installation des bibliothèques

Exécutez cette cellule une seule fois si les bibliothèques ne sont pas encore installées.

In [ ]:
%pip install -q yfinance pandas numpy matplotlib seaborn

## 2. Importation des bibliothèques et configuration

Nous utilisons `yfinance` pour télécharger les prix historiques, `pandas` et `numpy` pour les calculs, et `matplotlib`/`seaborn` pour les graphiques.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

TICKER = "AAPL"
DATE_DEBUT = "2015-01-01"
DATE_FIN = None  # None = jusqu'à la date la plus récente disponible
CAPITAL_INITIAL = 10_000.0
FRAIS_PAR_TRANSACTION = 0.001  # 0,10 % à l'entrée et à la sortie
JOURS_PAR_AN = 252

print(f"Actif analysé : {TICKER}")
print(f"Période de début : {DATE_DEBUT}")
print(f"Capital initial : {CAPITAL_INITIAL:,.2f} $")

## 3. Téléchargement des données historiques AAPL

Les prix sont ajustés (`auto_adjust=True`) afin de prendre en compte les divisions d’actions et les dividendes dans la série de prix. Le téléchargement est effectué avec une seule source et la période exacte est affichée pour rendre l’analyse reproductible.

In [ ]:
data = yf.download(
    TICKER,
    start=DATE_DEBUT,
    end=DATE_FIN,
    auto_adjust=True,
    progress=False
)

if data.empty:
    raise ValueError("Aucune donnée téléchargée. Vérifiez la connexion ou le symbole boursier.")

# Selon la version de yfinance, les colonnes peuvent être simples ou multi-indexées.
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[["Open", "High", "Low", "Close", "Volume"]].copy()
data.index = pd.to_datetime(data.index).tz_localize(None)
data = data.sort_index()

display(data.head())
print(f"Période réellement téléchargée : {data.index.min().date()} → {data.index.max().date()}")
print(f"Nombre d'observations : {len(data):,}")

## 4. Contrôle et nettoyage des données

Nous vérifions les dimensions, les types, les valeurs manquantes et les doublons de dates. Les lignes incomplètes sont supprimées avant le calcul des indicateurs.

In [ ]:
print("Dimensions initiales :", data.shape)
data.info()

print("\nValeurs manquantes par colonne :")
display(data.isna().sum().to_frame("Valeurs manquantes"))

print("Doublons d'index :", data.index.duplicated().sum())

data = data[~data.index.duplicated(keep="last")].dropna().copy()
print("Dimensions après nettoyage :", data.shape)

## 5. Rendements journaliers et aperçu du prix

Le rendement simple journalier mesure la variation relative du cours de clôture d’un jour à l’autre. La première observation est naturellement manquante.

In [ ]:
data["Rendement"] = data["Close"].pct_change()

data[["Close", "Volume", "Rendement"]].tail()

## 6. Moyennes mobiles MA_100 et MA_200

La **MA_100** mesure la moyenne des 100 dernières clôtures. La **MA_200** mesure la moyenne des 200 dernières clôtures. La relation entre les deux donne une indication de tendance, mais elle ne prédit pas les rendements futurs.

In [ ]:
data["MA_100"] = data["Close"].rolling(window=100, min_periods=100).mean()
data["MA_200"] = data["Close"].rolling(window=200, min_periods=200).mean()

data[["Close", "MA_100", "MA_200"]].tail()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data["Close"], label="AAPL - clôture", linewidth=1.2)
plt.plot(data["MA_100"], label="MA_100", linewidth=1.2)
plt.plot(data["MA_200"], label="MA_200", linewidth=1.2)
plt.title("AAPL : prix de clôture et moyennes mobiles")
plt.xlabel("Date")
plt.ylabel("Prix ajusté ($)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. RSI (Relative Strength Index)

Le **RSI** est un oscillateur compris entre 0 et 100. Ici, il est calculé sur 14 séances avec une moyenne exponentielle. Des valeurs élevées indiquent une hausse récente forte et des valeurs faibles une baisse récente forte. Les seuils 30/70 sont des repères, pas des garanties de retournement.

In [ ]:
PERIODE_RSI = 14

delta = data["Close"].diff()
gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)
avg_gain = gains.ewm(alpha=1 / PERIODE_RSI, adjust=False, min_periods=PERIODE_RSI).mean()
avg_loss = losses.ewm(alpha=1 / PERIODE_RSI, adjust=False, min_periods=PERIODE_RSI).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
data["RSI_14"] = 100 - (100 / (1 + rs))
data["RSI_14"] = data["RSI_14"].fillna(100).clip(0, 100)

data[["Close", "RSI_14"]].tail()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax[0].plot(data["Close"], label="AAPL - clôture")
ax[0].set_title("Prix de clôture")
ax[0].legend()
ax[1].plot(data["RSI_14"], color="purple", label="RSI 14")
ax[1].axhline(70, color="red", linestyle="--", label="Seuil 70")
ax[1].axhline(30, color="green", linestyle="--", label="Seuil 30")
ax[1].set_ylim(0, 100)
ax[1].set_title("RSI (14 séances)")
ax[1].legend()
plt.tight_layout()
plt.show()

## 8. MACD

Le **MACD** compare deux moyennes mobiles exponentielles : 12 et 26 séances. La ligne de signal est une moyenne exponentielle à 9 séances du MACD. Un MACD supérieur à sa ligne de signal indique une dynamique haussière relative selon cette règle.

In [ ]:
EMA_RAPIDE = 12
EMA_LENTE = 26
PERIODE_SIGNAL = 9

data["EMA_12"] = data["Close"].ewm(span=EMA_RAPIDE, adjust=False).mean()
data["EMA_26"] = data["Close"].ewm(span=EMA_LENTE, adjust=False).mean()
data["MACD"] = data["EMA_12"] - data["EMA_26"]
data["MACD_Signal"] = data["MACD"].ewm(span=PERIODE_SIGNAL, adjust=False).mean()
data["MACD_Histogramme"] = data["MACD"] - data["MACD_Signal"]

data[["Close", "MACD", "MACD_Signal", "MACD_Histogramme"]].tail()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax[0].plot(data["Close"], label="AAPL - clôture")
ax[0].set_title("Prix de clôture")
ax[0].legend()
ax[1].plot(data["MACD"], label="MACD", color="blue")
ax[1].plot(data["MACD_Signal"], label="Ligne de signal", color="orange")
ax[1].bar(data.index, data["MACD_Histogramme"], label="Histogramme", alpha=0.35)
ax[1].axhline(0, color="black", linewidth=0.8)
ax[1].set_title("MACD (12, 26, 9)")
ax[1].legend()
plt.tight_layout()
plt.show()

## 9. Volatilité historique annualisée

La volatilité est l’écart-type des rendements journaliers sur une fenêtre glissante de 20 séances, annualisé par √252. Elle mesure l’amplitude historique des variations, mais ne prédit pas directement le sens du prochain mouvement.

In [ ]:
FENETRE_VOL = 20
data["Volatilite_20j"] = data["Rendement"].rolling(FENETRE_VOL).std() * np.sqrt(JOURS_PAR_AN)

data[["Rendement", "Volatilite_20j"]].tail()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(data["Volatilite_20j"] * 100, color="darkorange")
plt.title("Volatilité historique annualisée sur 20 séances")
plt.ylabel("Volatilité (%)")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

## 10. Analyse du volume

Le volume est comparé à sa moyenne mobile sur 20 séances. Le filtre `Volume > Volume_MA_20` retient uniquement les séances où les échanges sont supérieurs à leur niveau moyen récent.

In [ ]:
data["Volume_MA_20"] = data["Volume"].rolling(20).mean()
data["Ratio_Volume"] = data["Volume"] / data["Volume_MA_20"]

data[["Volume", "Volume_MA_20", "Ratio_Volume"]].tail()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax[0].plot(data["Close"], color="black")
ax[0].set_title("AAPL - prix de clôture")
ax[1].bar(data.index, data["Volume"], alpha=0.45, label="Volume")
ax[1].plot(data["Volume_MA_20"], color="red", label="Moyenne du volume (20)")
ax[1].set_title("Volume quotidien et moyenne mobile du volume")
ax[1].legend()
plt.tight_layout()
plt.show()

## 11. Définition de la stratégie

La stratégie est **long-only** et utilise les règles suivantes :

- tendance haussière : `MA_100 > MA_200` ;
- momentum haussier : `MACD > MACD_Signal` ;
- RSI non situé en zone de surachat : `40 <= RSI <= 70` ;
- volatilité non excessive : volatilité inférieure ou égale à son 75e percentile historique ;
- confirmation par le volume : volume supérieur à sa moyenne sur 20 séances.

La position est calculée à partir du signal de la veille (`shift(1)`) afin d’éviter d’utiliser la clôture du jour pour simuler une exécution au même prix.

In [ ]:
# Seuil de volatilité établi sur les observations disponibles.
SEUIL_VOLATILITE = data["Volatilite_20j"].quantile(0.75)

condition_tendance = data["MA_100"] > data["MA_200"]
condition_macd = data["MACD"] > data["MACD_Signal"]
condition_rsi = data["RSI_14"].between(40, 70)
condition_volatilite = data["Volatilite_20j"] <= SEUIL_VOLATILITE
condition_volume = data["Volume"] > data["Volume_MA_20"]

data["Signal"] = (
    condition_tendance
    & condition_macd
    & condition_rsi
    & condition_volatilite
    & condition_volume
).astype(int)

# Décalage d'une séance : le signal devient une position le jour suivant.
data["Position"] = data["Signal"].shift(1).fillna(0)

print(f"Seuil de volatilité utilisé : {SEUIL_VOLATILITE:.2%}")
print(f"Nombre de séances en position : {int(data['Position'].sum()):,}")
data[["Close", "Signal", "Position"]].tail(10)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data["Close"], label="AAPL - clôture", color="black")
plt.fill_between(
    data.index,
    data["Close"].min(),
    data["Close"].max(),
    where=data["Position"].eq(1),
    color="green",
    alpha=0.15,
    label="Position acheteuse"
)
plt.title("Zones de détention de la stratégie")
plt.ylabel("Prix ajusté ($)")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Backtesting avec frais de transaction

Le rendement de la stratégie est le rendement de l’action multiplié par la position décalée. Un coût de `0,10 %` est appliqué à chaque changement de position, à l’entrée comme à la sortie. Ce modèle reste simplifié : il ne représente pas les écarts achat/vente, les impôts, les dividendes non réinvestis séparément ou les contraintes de liquidité.

In [ ]:
data["Rendement_Strategie_Brut"] = data["Position"] * data["Rendement"]
data["Variation_Position"] = data["Position"].diff().abs().fillna(0)
data["Frais"] = data["Variation_Position"] * FRAIS_PAR_TRANSACTION
data["Rendement_Strategie"] = data["Rendement_Strategie_Brut"] - data["Frais"]

data["Capital_Strategie"] = CAPITAL_INITIAL * (1 + data["Rendement_Strategie"].fillna(0)).cumprod()
data["Capital_Buy_Hold"] = CAPITAL_INITIAL * (1 + data["Rendement"].fillna(0)).cumprod()

data[["Position", "Rendement_Strategie", "Frais", "Capital_Strategie"]].tail()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data["Capital_Strategie"], label="Stratégie", linewidth=1.5)
plt.plot(data["Capital_Buy_Hold"], label="Buy & Hold AAPL", linewidth=1.5)
plt.axhline(CAPITAL_INITIAL, color="gray", linestyle="--", linewidth=0.8)
plt.title("Évolution d'un capital initial de 10 000 $")
plt.ylabel("Capital ($)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Calcul du rendement total, du rendement annualisé et du Sharpe Ratio

- Le **rendement total** compare le capital final au capital initial.
- Le **rendement annualisé** annualise la croissance composée sur la durée observée.
- Le **Sharpe Ratio** est calculé ici avec un taux sans risque supposé nul. Il compare le rendement moyen annualisé à la volatilité annualisée. Ce choix doit être documenté si l’on utilise ce résultat dans une analyse plus approfondie.

In [ ]:
def calcul_metriques(serie_rendements, capital):
    rendements = serie_rendements.dropna()
    capital = capital.dropna()
    nb_annees = len(rendements) / JOURS_PAR_AN
    rendement_total = capital.iloc[-1] / capital.iloc[0] - 1
    rendement_annualise = (capital.iloc[-1] / capital.iloc[0]) ** (1 / nb_annees) - 1 if nb_annees > 0 else np.nan
    volatilite_annualisee = rendements.std() * np.sqrt(JOURS_PAR_AN)
    sharpe = (rendements.mean() / rendements.std()) * np.sqrt(JOURS_PAR_AN) if rendements.std() != 0 else np.nan
    return {
        "Capital final ($)": capital.iloc[-1],
        "Rendement total (%)": rendement_total * 100,
        "Rendement annualisé (%)": rendement_annualise * 100,
        "Volatilité annualisée (%)": volatilite_annualisee * 100,
        "Sharpe Ratio": sharpe,
    }

metriques_strategie = calcul_metriques(data["Rendement_Strategie"], data["Capital_Strategie"])
metriques_buy_hold = calcul_metriques(data["Rendement"], data["Capital_Buy_Hold"])

comparaison = pd.DataFrame(
    [metriques_strategie, metriques_buy_hold],
    index=["Stratégie", "Buy & Hold"]
)
display(comparaison.round(3))

## 14. Maximum Drawdown

Le **Maximum Drawdown** mesure la baisse maximale entre un sommet historique du capital et le creux qui suit. Il décrit le risque de perte depuis un précédent sommet, sans supposer que la perte est réalisée.

In [ ]:
def calcul_drawdown(capital):
    sommet_roulant = capital.cummax()
    drawdown = capital / sommet_roulant - 1
    maximum_drawdown = drawdown.min()
    date_creux = drawdown.idxmin()
    date_sommet = capital.loc[:date_creux].idxmax()
    return maximum_drawdown, date_sommet, date_creux, drawdown

mdd_strategie, sommet_strategie, creux_strategie, drawdown_strategie = calcul_drawdown(data["Capital_Strategie"])
mdd_buy_hold, sommet_buy_hold, creux_buy_hold, drawdown_buy_hold = calcul_drawdown(data["Capital_Buy_Hold"])

print(f"Maximum Drawdown - Stratégie : {mdd_strategie:.2%}")
print(f"Période du drawdown : {sommet_strategie.date()} → {creux_strategie.date()}")
print(f"Maximum Drawdown - Buy & Hold : {mdd_buy_hold:.2%}")
print(f"Période du drawdown : {sommet_buy_hold.date()} → {creux_buy_hold.date()}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(drawdown_strategie * 100, label="Drawdown stratégie")
plt.plot(
    (data["Capital_Buy_Hold"] / data["Capital_Buy_Hold"].cummax() - 1) * 100,
    label="Drawdown Buy & Hold",
    alpha=0.8
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Drawdown historique")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Résumé de l’analyse et limites

Ce notebook fournit une simulation reproductible d’une stratégie technique long-only sur AAPL. Les métriques doivent être interprétées avec prudence : le modèle utilise un seul actif, une fréquence quotidienne, des paramètres fixes et un modèle de frais simplifié. Il ne réalise pas de test hors échantillon, d’analyse de sensibilité des paramètres ni de simulation de glissement de prix.

Pour améliorer la robustesse, il faudrait séparer les périodes d’entraînement et de test, comparer plusieurs combinaisons de paramètres sans sélectionner uniquement la meilleure, intégrer un benchmark cohérent et tester plusieurs régimes de marché.

In [ ]:
print("=" * 60)
print(f"PIPELINE D'ANALYSE - {TICKER}")
print("=" * 60)
print("Données historiques téléchargées et nettoyées          ✓")
print("MA_100 et MA_200 calculées                             ✓")
print("RSI calculé                                             ✓")
print("MACD calculé                                            ✓")
print("Volatilité et volume analysés                           ✓")
print("Stratégie définie avec position décalée                ✓")
print("Backtest avec frais de transaction                     ✓")
print("Rendement, Sharpe Ratio et Maximum Drawdown calculés   ✓")
print("=" * 60)